# Somo la 04 - Mfano wa Ubunifu wa Matumizi ya Zana

Katika somo hili utajifunza **Mfano wa Ubunifu wa Matumizi ya Zana** kwa maajenti wa AI ukitumia Microsoft Agent Framework (Python). Tunajumuisha:

- Kueleza zana za kazi kwa kutumia dekoreta `@tool` na vigezo vilivyotumika aina
- Kutoa miundo ya zana ili mfano ujue kila zana inafanya nini
- Kudhibiti utekelezaji wa zana kwa kutumia `approval_mode`
- Kurudisha **matokeo yaliyopangwa** kupitia mifano ya Pydantic na `response_format`

Muktadha ni wakala wa **kuandaa safari** ambaye anaweza kutafuta maeneo, kuangalia upatikanaji, na kupata taarifa za ndege.


## Mipangilio


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kuelezea Vifaa kwa Mtindo wa @tool

Mtindo wa `@tool` hubadilisha kazi ya kawaida ya Python kuwa kifaa ambacho wakombozi wanaweza kuituma.
Mambo muhimu:

- **Docstring** hubadilika kuwa maelezo ya kifaa ambayo mfano unaona.
- **Maelezo ya aina** (ikiwemo `Annotated` yenye maelezo) huelezea muundo wa kifaa.
- `approval_mode` hudhibiti kama mtumiaji lazima akuze kila simu kabla haitekelezwi.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Kuunda Wakala na Vifaa Vingi

Pitia zana zote tatu kwa mteja ili mfano uweze kuitumia yoyote ile inayo hitajika kujibu swali la mtumiaji.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Matokeo Yaliyoandaliwa kwa Vifaa

Kwa kuweka `response_format` kuwa mfano wa Pydantic, wakala anasukumwa kurudisha vitu vya JSON vilivyo na aina maalum badala ya maandishi ya aina yoyote. Hii ni muhimu wakati msimbo unaofuata unahitaji kutumia matokeo kwa njia ya programu.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Mifumo ya Uidhinishaji wa Zana

Kiparameta cha `approval_mode` kwenye `@tool` hudhibiti kama simu za zana zinahitaji idhini ya binadamu kabla ya kutekelezwa:

| Mode | Tabia |
|---|---|
| `"never_require"` | Zana zinaendesha moja kwa moja — hakuna uthibitisho wa mtumiaji unaohitajika. |
| `"always_require"` | Kila simu lazima idhinishwe na mtumiaji kabla ya kutekelezwa. |

Tumia `"always_require"` kwa zana zinazokuwa na athari za pembeni (kwa mfano kuhifadhi tiketi ya ndege, kutoza kadi ya mkopo) ili binadamu abaki katika mzunguko.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Kufafanua zana** kwa kutumia dekoreta `@tool` pamoja na vigezo vyenye aina na docstrings vinavyotumika kama mfumo wa zana.
2. **Kuunganisha zana nyingi** ili wakala aweze kuziita mfululizo kujibu maswali magumu.
3. **Kurudisha matokeo yenye muundo** kwa kupitisha mfano wa Pydantic kama `response_format`.
4. **Kudhibiti idhini ya zana** kwa `approval_mode` ili kumweka binadamu katika mzunguko kwa shughuli nyeti.

Mifumo hii huunda msingi wa kujenga mawakala wa kuaminika, tayari kwa uzalishaji ambao wanaweza kuwasiliana na mifumo ya nje kwa usalama.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
